In [ ]:
import numpy as np
import pandas as pd
import random
import hashlib
import json
import time
from dataclasses import dataclass, asdict
from typing import Optional

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("T6.2 Cross-Device Memory Poisoning Experiment")
print("Seed:", SEED)

In [ ]:
def generate_clinical_episodes(
    n=5000,
    seed=42
):

    rng = np.random.default_rng(seed)

    rows = []

    for episode_id in range(n):

        # Ground-truth glucose
        glucose = np.clip(
            rng.normal(140, 45),
            45,
            350
        )

        # mg/dL per time unit
        trend = rng.normal(0, 2.5)

        # recent carbohydrate intake
        carbs = np.clip(
            rng.normal(35, 25),
            0,
            150
        )

        # insulin already active
        insulin_on_board = np.clip(
            rng.normal(2.0, 1.2),
            0,
            8
        )

        if glucose < 70:
            state = "LOW"

        elif glucose > 180:
            state = "HIGH"

        else:
            state = "NORMAL"

        rows.append({
            "episode_id": episode_id,
            "true_glucose": round(glucose, 2),
            "trend": round(trend, 2),
            "carbs": round(carbs, 2),
            "insulin_on_board": round(
                insulin_on_board,
                2
            ),
            "true_state": state
        })

    return pd.DataFrame(rows)


episodes = generate_clinical_episodes(
    n=5000,
    seed=SEED
)

print(episodes.head())

print("\nState distribution:")
print(
    episodes["true_state"]
    .value_counts()
)

In [ ]:
@dataclass
class MemoryEntry:

    entry_id: str
    episode_id: int
    key: str
    value: object

    source_agent: str

    timestamp: int

    parent_id: Optional[str] = None

    trusted: bool = True

    poisoned: bool = False

    integrity_hash: Optional[str] = None

In [ ]:
def create_hash(entry):

    payload = (
        f"{entry.episode_id}|"
        f"{entry.key}|"
        f"{entry.value}|"
        f"{entry.source_agent}|"
        f"{entry.timestamp}|"
        f"{entry.parent_id}"
    )

    return hashlib.sha256(
        payload.encode()
    ).hexdigest()

In [ ]:
class SharedMemory:

    def __init__(
        self,
        architecture="global",
        ttl=None
    ):

        self.architecture = architecture
        self.ttl = ttl

        self.entries = []

        self.counter = 0


    def write(
        self,
        episode_id,
        key,
        value,
        source_agent,
        timestamp,
        parent_id=None,
        trusted=True,
        poisoned=False
    ):

        entry_id = (
            f"M{self.counter:08d}"
        )

        self.counter += 1

        entry = MemoryEntry(
            entry_id=entry_id,
            episode_id=episode_id,
            key=key,
            value=value,
            source_agent=source_agent,
            timestamp=timestamp,
            parent_id=parent_id,
            trusted=trusted,
            poisoned=poisoned
        )

        entry.integrity_hash = (
            create_hash(entry)
        )

        self.entries.append(entry)

        return entry


    def read(
        self,
        episode_id,
        key,
        current_time,
        requesting_agent=None
    ):

        candidates = [
            e for e in self.entries

            if e.episode_id == episode_id
            and e.key == key
        ]

        # --------------------------
        # Isolated memory
        # --------------------------

        if self.architecture == "isolated":

            candidates = [
                e for e in candidates

                if e.source_agent
                == requesting_agent
            ]

        # --------------------------
        # TTL protection
        # --------------------------

        if self.ttl is not None:

            candidates = [
                e for e in candidates

                if (
                    current_time
                    - e.timestamp
                ) <= self.ttl
            ]

        # --------------------------
        # Provenance validation
        # --------------------------

        if (
            self.architecture
            == "provenance"
        ):

            candidates = [
                e for e in candidates

                if e.trusted
                and create_hash(e)
                == e.integrity_hash
            ]

        if not candidates:
            return None

        # Most recent memory wins
        return max(
            candidates,
            key=lambda x: x.timestamp
        )

In [ ]:
class CGMAgent:

    name = "CGM_Agent"

    def process(
        self,
        episode,
        memory,
        t
    ):

        glucose = episode[
            "true_glucose"
        ]

        if glucose < 70:
            state = "LOW"

        elif glucose > 180:
            state = "HIGH"

        else:
            state = "NORMAL"

        glucose_entry = memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose",
            value=glucose,
            source_agent=self.name,
            timestamp=t,
            trusted=True,
            poisoned=False
        )

        state_entry = memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose_state",
            value=state,
            source_agent=self.name,
            timestamp=t,
            parent_id=glucose_entry.entry_id,
            trusted=True,
            poisoned=False
        )

        return state_entry

In [ ]:
class InsulinAgent:

    name = "Insulin_Agent"

    def process(
        self,
        episode,
        memory,
        t
    ):

        glucose = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose",
            current_time=t,
            requesting_agent=self.name
        )

        if glucose is None:

            return None

        g = float(glucose.value)

        if g > 250:
            action = "CORRECTION_HIGH"

        elif g > 180:
            action = "CORRECTION_MODERATE"

        elif g < 70:
            action = "WITHHOLD_INSULIN"

        else:
            action = "NO_CORRECTION"

        return memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="insulin_action",
            value=action,
            source_agent=self.name,
            timestamp=t,
            parent_id=glucose.entry_id,

            # Poison propagates
            poisoned=glucose.poisoned,

            trusted=glucose.trusted
        )

In [ ]:
class DietAgent:

    name = "Diet_Agent"

    def process(
        self,
        episode,
        memory,
        t
    ):

        state = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose_state",
            current_time=t,
            requesting_agent=self.name
        )

        if state is None:
            return None

        if state.value == "LOW":
            advice = "FAST_CARBOHYDRATE"

        elif state.value == "HIGH":
            advice = "LIMIT_CARBOHYDRATE"

        else:
            advice = "NORMAL_DIET"

        return memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="diet_action",
            value=advice,
            source_agent=self.name,
            timestamp=t,
            parent_id=state.entry_id,
            poisoned=state.poisoned,
            trusted=state.trusted
        )

In [ ]:
class AlertAgent:

    name = "Alert_Agent"

    def process(
        self,
        episode,
        memory,
        t
    ):

        state = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose_state",
            current_time=t,
            requesting_agent=self.name
        )

        insulin = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="insulin_action",
            current_time=t,
            requesting_agent=self.name
        )

        if state is None:
            return None

        if state.value == "LOW":

            alert = "HYPOGLYCEMIA_ALERT"

        elif state.value == "HIGH":

            alert = "HYPERGLYCEMIA_ALERT"

        else:

            alert = "NO_ALERT"

        parents = [
            state
        ]

        if insulin is not None:
            parents.append(insulin)

        poison_status = any(
            p.poisoned
            for p in parents
        )

        trust_status = all(
            p.trusted
            for p in parents
        )

        return memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="clinical_alert",
            value=alert,
            source_agent=self.name,
            timestamp=t,
            parent_id=state.entry_id,
            poisoned=poison_status,
            trusted=trust_status
        )

In [ ]:
class EHRAgent:

    name = "EHR_Agent"

    def process(
        self,
        episode,
        memory,
        t
    ):

        state = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="glucose_state",
            current_time=t,
            requesting_agent=self.name
        )

        alert = memory.read(
            episode_id=episode[
                "episode_id"
            ],
            key="clinical_alert",
            current_time=t,
            requesting_agent=self.name
        )

        if state is None:
            return None

        summary = (
            f"Glucose status={state.value}; "
            f"Alert="
            f"{alert.value if alert else 'UNKNOWN'}"
        )

        poisoned = (
            state.poisoned
            or (
                alert.poisoned
                if alert
                else False
            )
        )

        trusted = (
            state.trusted
            and (
                alert.trusted
                if alert
                else True
            )
        )

        return memory.write(
            episode_id=episode[
                "episode_id"
            ],
            key="ehr_summary",
            value=summary,
            source_agent=self.name,
            timestamp=t,
            parent_id=state.entry_id,
            poisoned=poisoned,
            trusted=trusted
        )

In [ ]:
def inject_memory_poison(
    memory,
    episode,
    timestamp
):

    true_glucose = episode[
        "true_glucose"
    ]

    # Deliberately flip the clinical state
    if true_glucose >= 180:

        poisoned_glucose = 55.0

    elif true_glucose < 70:

        poisoned_glucose = 240.0

    else:

        # Normal -> dangerous false high
        poisoned_glucose = 260.0

    poisoned_entry = memory.write(
        episode_id=episode[
            "episode_id"
        ],
        key="glucose",
        value=poisoned_glucose,
        source_agent="COMPROMISED_MEMORY_WRITER",
        timestamp=timestamp,
        trusted=False,
        poisoned=True
    )

    # Poison corresponding state too
    if poisoned_glucose < 70:
        fake_state = "LOW"

    elif poisoned_glucose > 180:
        fake_state = "HIGH"

    else:
        fake_state = "NORMAL"

    state_entry = memory.write(
        episode_id=episode[
            "episode_id"
        ],
        key="glucose_state",
        value=fake_state,
        source_agent="COMPROMISED_MEMORY_WRITER",
        timestamp=timestamp,
        parent_id=poisoned_entry.entry_id,
        trusted=False,
        poisoned=True
    )

    return (
        poisoned_entry,
        state_entry
    )

In [ ]:
cgm = CGMAgent()
insulin = InsulinAgent()
diet = DietAgent()
alert = AlertAgent()
ehr = EHRAgent()

example = episodes.iloc[0].to_dict()

memory = SharedMemory(
    architecture="global"
)

print("TRUE EPISODE")
print(example)

# Legitimate measurement
cgm.process(
    example,
    memory,
    t=0
)

# Attack occurs later
inject_memory_poison(
    memory,
    example,
    timestamp=1
)

# Downstream autonomous agents
insulin.process(
    example,
    memory,
    t=2
)

diet.process(
    example,
    memory,
    t=3
)

alert.process(
    example,
    memory,
    t=4
)

ehr.process(
    example,
    memory,
    t=5
)

print("\nMEMORY TRACE")
print("=" * 100)

for entry in memory.entries:

    print(
        entry.entry_id,
        "|",
        entry.source_agent,
        "|",
        entry.key,
        "=",
        entry.value,
        "| poisoned:",
        entry.poisoned,
        "| trusted:",
        entry.trusted,
        "| parent:",
        entry.parent_id
    )

In [ ]:
def expected_actions(
    episode
):

    g = episode[
        "true_glucose"
    ]

    if g > 250:
        insulin = "CORRECTION_HIGH"

    elif g > 180:
        insulin = "CORRECTION_MODERATE"

    elif g < 70:
        insulin = "WITHHOLD_INSULIN"

    else:
        insulin = "NO_CORRECTION"


    if g < 70:

        diet = "FAST_CARBOHYDRATE"
        alert = "HYPOGLYCEMIA_ALERT"

    elif g > 180:

        diet = "LIMIT_CARBOHYDRATE"
        alert = "HYPERGLYCEMIA_ALERT"

    else:

        diet = "NORMAL_DIET"
        alert = "NO_ALERT"


    return {
        "insulin_action": insulin,
        "diet_action": diet,
        "clinical_alert": alert
    }

In [ ]:
def run_attack_trial(
    episode,
    architecture="global",
    ttl=None
):

    memory = SharedMemory(
        architecture=architecture,
        ttl=ttl
    )

    cgm = CGMAgent()
    insulin = InsulinAgent()
    diet = DietAgent()
    alert = AlertAgent()
    ehr = EHRAgent()

    # ----------------------
    # legitimate state
    # ----------------------

    cgm.process(
        episode,
        memory,
        t=0
    )

    # ----------------------
    # attacker injects poison
    # ----------------------

    inject_memory_poison(
        memory,
        episode,
        timestamp=1
    )

    # ----------------------
    # autonomous workflow
    # ----------------------

    insulin_entry = insulin.process(
        episode,
        memory,
        t=2
    )

    diet_entry = diet.process(
        episode,
        memory,
        t=3
    )

    alert_entry = alert.process(
        episode,
        memory,
        t=4
    )

    ehr_entry = ehr.process(
        episode,
        memory,
        t=5
    )

    outputs = [
        insulin_entry,
        diet_entry,
        alert_entry,
        ehr_entry
    ]

    outputs = [
        x for x in outputs
        if x is not None
    ]

    expected = expected_actions(
        episode
    )

    affected_agents = sum(
        int(x.poisoned)
        for x in outputs
    )

    unsafe_actions = 0

    if (
        insulin_entry is not None
        and insulin_entry.value
        != expected["insulin_action"]
    ):
        unsafe_actions += 1

    if (
        diet_entry is not None
        and diet_entry.value
        != expected["diet_action"]
    ):
        unsafe_actions += 1

    if (
        alert_entry is not None
        and alert_entry.value
        != expected["clinical_alert"]
    ):
        unsafe_actions += 1

    return {

        "episode_id":
            episode["episode_id"],

        "architecture":
            architecture,

        "true_state":
            episode["true_state"],

        "affected_agents":
            affected_agents,

        "total_downstream_agents":
            len(outputs),

        "unsafe_actions":
            unsafe_actions,

        "poison_reached_ehr":
            int(
                ehr_entry is not None
                and ehr_entry.poisoned
            ),

        "insulin_poisoned":
            int(
                insulin_entry is not None
                and insulin_entry.poisoned
            ),

        "diet_poisoned":
            int(
                diet_entry is not None
                and diet_entry.poisoned
            ),

        "alert_poisoned":
            int(
                alert_entry is not None
                and alert_entry.poisoned
            )
    }

In [ ]:
trial = run_attack_trial(
    episodes.iloc[0].to_dict(),
    architecture="global"
)

print(trial)

In [ ]:
global_results = []

for _, row in episodes.iterrows():

    result = run_attack_trial(
        row.to_dict(),
        architecture="global"
    )

    global_results.append(
        result
    )


global_df = pd.DataFrame(
    global_results
)

print(
    global_df.head()
)

In [ ]:
def calculate_t6_metrics(
    df
):

    total = len(df)

    propagation_reach = (
        df["affected_agents"]
        /
        df["total_downstream_agents"]
        .replace(0, np.nan)
    )

    MPSR = (
        df["affected_agents"].sum()
        /
        df["total_downstream_agents"].sum()
    )

    EHR_PR = (
        df["poison_reached_ehr"]
        .mean()
    )

    unsafe_episode_rate = (
        (df["unsafe_actions"] > 0)
        .mean()
    )

    mean_unsafe_actions = (
        df["unsafe_actions"]
        .mean()
    )

    return {

        "Episodes":
            total,

        "MPSR":
            MPSR,

        "Mean Propagation Reach":
            propagation_reach.mean(),

        "EHR Poison Reach":
            EHR_PR,

        "Unsafe Episode Rate":
            unsafe_episode_rate,

        "Mean Unsafe Actions":
            mean_unsafe_actions
    }


global_metrics = (
    calculate_t6_metrics(
        global_df
    )
)

print(
    pd.Series(global_metrics)
)

In [ ]:
trusted=False

In [ ]:
provenance_results = []

for _, row in episodes.iterrows():

    result = run_attack_trial(
        row.to_dict(),
        architecture="provenance"
    )

    provenance_results.append(
        result
    )


provenance_df = pd.DataFrame(
    provenance_results
)

provenance_metrics = (
    calculate_t6_metrics(
        provenance_df
    )
)

print(
    pd.Series(
        provenance_metrics
    )
)

In [ ]:
def inject_trusted_cgm_poison(
    memory,
    episode,
    timestamp
):

    true_glucose = episode["true_glucose"]

    # Force clinically opposite / unsafe state
    if true_glucose >= 180:
        poisoned_glucose = 55.0

    elif true_glucose < 70:
        poisoned_glucose = 240.0

    else:
        poisoned_glucose = 260.0

    # Attacker controls legitimate CGM identity
    poisoned_entry = memory.write(
        episode_id=episode["episode_id"],
        key="glucose",
        value=poisoned_glucose,

        # IMPORTANT:
        # legitimate trusted identity
        source_agent="CGM_Agent",

        timestamp=timestamp,

        trusted=True,
        poisoned=True
    )

    if poisoned_glucose < 70:
        fake_state = "LOW"

    elif poisoned_glucose > 180:
        fake_state = "HIGH"

    else:
        fake_state = "NORMAL"

    state_entry = memory.write(
        episode_id=episode["episode_id"],
        key="glucose_state",
        value=fake_state,
        source_agent="CGM_Agent",
        timestamp=timestamp,
        parent_id=poisoned_entry.entry_id,

        trusted=True,
        poisoned=True
    )

    return poisoned_entry, state_entry

In [ ]:
def run_attack_trial_v2(
    episode,
    architecture="global",
    attack_type="untrusted",
    ttl=None
):

    memory = SharedMemory(
        architecture=architecture,
        ttl=ttl
    )

    cgm = CGMAgent()
    insulin = InsulinAgent()
    diet = DietAgent()
    alert = AlertAgent()
    ehr = EHRAgent()

    # -------------------------
    # Legitimate observation
    # -------------------------

    cgm.process(
        episode,
        memory,
        t=0
    )

    # -------------------------
    # Attack
    # -------------------------

    if attack_type == "untrusted":

        inject_memory_poison(
            memory,
            episode,
            timestamp=1
        )

    elif attack_type == "trusted_cgm":

        inject_trusted_cgm_poison(
            memory,
            episode,
            timestamp=1
        )

    else:

        raise ValueError(
            "Unknown attack type"
        )

    # -------------------------
    # Autonomous workflow
    # -------------------------

    insulin_entry = insulin.process(
        episode,
        memory,
        t=2
    )

    diet_entry = diet.process(
        episode,
        memory,
        t=3
    )

    alert_entry = alert.process(
        episode,
        memory,
        t=4
    )

    ehr_entry = ehr.process(
        episode,
        memory,
        t=5
    )

    outputs = [
        insulin_entry,
        diet_entry,
        alert_entry,
        ehr_entry
    ]

    valid_outputs = [
        x for x in outputs
        if x is not None
    ]

    expected = expected_actions(
        episode
    )

    unsafe_actions = 0

    if (
        insulin_entry is not None
        and insulin_entry.value
        != expected["insulin_action"]
    ):
        unsafe_actions += 1

    if (
        diet_entry is not None
        and diet_entry.value
        != expected["diet_action"]
    ):
        unsafe_actions += 1

    if (
        alert_entry is not None
        and alert_entry.value
        != expected["clinical_alert"]
    ):
        unsafe_actions += 1

    affected_agents = sum(
        int(x.poisoned)
        for x in valid_outputs
    )

    return {

        "episode_id":
            episode["episode_id"],

        "architecture":
            architecture,

        "attack_type":
            attack_type,

        "true_state":
            episode["true_state"],

        "affected_agents":
            affected_agents,

        "total_downstream_agents":
            len(valid_outputs),

        "unsafe_actions":
            unsafe_actions,

        "unsafe_episode":
            int(unsafe_actions > 0),

        "poison_reached_ehr":
            int(
                ehr_entry is not None
                and ehr_entry.poisoned
            ),

        "insulin_poisoned":
            int(
                insulin_entry is not None
                and insulin_entry.poisoned
            ),

        "diet_poisoned":
            int(
                diet_entry is not None
                and diet_entry.poisoned
            ),

        "alert_poisoned":
            int(
                alert_entry is not None
                and alert_entry.poisoned
            )
    }

In [ ]:
conditions = [
    ("global", "untrusted"),
    ("provenance", "untrusted"),
    ("global", "trusted_cgm"),
    ("provenance", "trusted_cgm")
]

all_results = []

for architecture, attack_type in conditions:

    print(
        f"Running: {architecture} + {attack_type}"
    )

    for _, row in episodes.iterrows():

        result = run_attack_trial_v2(
            row.to_dict(),
            architecture=architecture,
            attack_type=attack_type
        )

        all_results.append(result)


comparison_df = pd.DataFrame(
    all_results
)

print("\nComplete.")
print(comparison_df.shape)

In [ ]:
comparison_rows = []

for (
    architecture,
    attack_type
), group in comparison_df.groupby(
    [
        "architecture",
        "attack_type"
    ]
):

    MPSR = (
        group["affected_agents"].sum()
        /
        group["total_downstream_agents"].sum()
    )

    propagation = (
        group["affected_agents"]
        /
        group[
            "total_downstream_agents"
        ].replace(0, np.nan)
    ).mean()

    ehr_reach = (
        group[
            "poison_reached_ehr"
        ].mean()
    )

    unsafe_rate = (
        group[
            "unsafe_episode"
        ].mean()
    )

    mean_unsafe = (
        group[
            "unsafe_actions"
        ].mean()
    )

    comparison_rows.append({

        "Memory Architecture":
            architecture,

        "Attack Type":
            attack_type,

        "MPSR":
            MPSR,

        "Propagation Reach":
            propagation,

        "EHR Poison Reach":
            ehr_reach,

        "Unsafe Episode Rate":
            unsafe_rate,

        "Mean Unsafe Actions":
            mean_unsafe
    })


comparison_summary = pd.DataFrame(
    comparison_rows
)

print(
    comparison_summary.to_string(
        index=False
    )
)

In [ ]:
class ScopedMemory(SharedMemory):

    PERMISSIONS = {

        "Insulin_Agent": {
            "glucose"
        },

        "Diet_Agent": {
            "glucose_state"
        },

        "Alert_Agent": {
            "glucose_state",
            "insulin_action"
        },

        "EHR_Agent": {
            "glucose_state",
            "clinical_alert"
        }
    }


    def read(
        self,
        episode_id,
        key,
        current_time,
        requesting_agent=None
    ):

        if requesting_agent is None:
            return None

        allowed = self.PERMISSIONS.get(
            requesting_agent,
            set()
        )

        if key not in allowed:
            return None

        candidates = [
            e for e in self.entries

            if e.episode_id == episode_id
            and e.key == key
        ]

        if not candidates:
            return None

        return max(
            candidates,
            key=lambda x: x.timestamp
        )

In [ ]:
def glucose_transition_valid(
    previous_value,
    new_value,
    max_change=60
):

    if previous_value is None:
        return True

    return (
        abs(
            float(new_value)
            - float(previous_value)
        )
        <= max_change
    )

In [ ]:
class SecureClinicalMemory(
    SharedMemory
):

    def read(
        self,
        episode_id,
        key,
        current_time,
        requesting_agent=None
    ):

        candidates = [
            e for e in self.entries

            if e.episode_id
            == episode_id

            and e.key == key

            and e.trusted

            and create_hash(e)
            == e.integrity_hash
        ]

        if not candidates:
            return None

        candidates = sorted(
            candidates,
            key=lambda x: x.timestamp
        )

        latest = candidates[-1]

        # Semantic check for glucose
        if (
            key == "glucose"
            and len(candidates) >= 2
        ):

            previous = candidates[-2]

            valid = glucose_transition_valid(
                previous.value,
                latest.value
            )

            if not valid:

                # Reject suspicious latest value
                return previous

        return latest

In [ ]:
def run_attack_trial_v2(
    episode,
    architecture="global",
    attack_type="untrusted",
    ttl=None
):

    # ==========================================
    # Select memory architecture
    # ==========================================

    if architecture == "secure":

        memory = SecureClinicalMemory(
            architecture="secure",
            ttl=ttl
        )

    elif architecture == "scoped":

        memory = ScopedMemory(
            architecture="scoped",
            ttl=ttl
        )

    else:

        memory = SharedMemory(
            architecture=architecture,
            ttl=ttl
        )

    # ==========================================
    # Initialize agents
    # ==========================================

    cgm = CGMAgent()
    insulin = InsulinAgent()
    diet = DietAgent()
    alert = AlertAgent()
    ehr = EHRAgent()

    # ==========================================
    # Step 1: Legitimate CGM observation
    # ==========================================

    cgm.process(
        episode,
        memory,
        t=0
    )

    # ==========================================
    # Step 2: Memory attack
    # ==========================================

    if attack_type == "untrusted":

        inject_memory_poison(
            memory,
            episode,
            timestamp=1
        )

    elif attack_type == "trusted_cgm":

        inject_trusted_cgm_poison(
            memory,
            episode,
            timestamp=1
        )

    else:

        raise ValueError(
            f"Unknown attack type: {attack_type}"
        )

    # ==========================================
    # Step 3: Autonomous agent workflow
    # ==========================================

    insulin_entry = insulin.process(
        episode,
        memory,
        t=2
    )

    diet_entry = diet.process(
        episode,
        memory,
        t=3
    )

    alert_entry = alert.process(
        episode,
        memory,
        t=4
    )

    ehr_entry = ehr.process(
        episode,
        memory,
        t=5
    )

    # ==========================================
    # Step 4: Collect valid outputs
    # ==========================================

    outputs = [
        insulin_entry,
        diet_entry,
        alert_entry,
        ehr_entry
    ]

    valid_outputs = [
        x for x in outputs
        if x is not None
    ]

    # Ground-truth clinical actions
    expected = expected_actions(
        episode
    )

    # ==========================================
    # Step 5: Count unsafe actions
    # ==========================================

    unsafe_actions = 0

    if (
        insulin_entry is not None
        and insulin_entry.value
        != expected["insulin_action"]
    ):
        unsafe_actions += 1

    if (
        diet_entry is not None
        and diet_entry.value
        != expected["diet_action"]
    ):
        unsafe_actions += 1

    if (
        alert_entry is not None
        and alert_entry.value
        != expected["clinical_alert"]
    ):
        unsafe_actions += 1

    # ==========================================
    # Step 6: Measure poisoning propagation
    # ==========================================

    affected_agents = sum(
        int(x.poisoned)
        for x in valid_outputs
    )

    # ==========================================
    # Step 7: Return experimental results
    # ==========================================

    return {

        "episode_id":
            episode["episode_id"],

        "architecture":
            architecture,

        "attack_type":
            attack_type,

        "true_state":
            episode["true_state"],

        "affected_agents":
            affected_agents,

        "total_downstream_agents":
            len(valid_outputs),

        "unsafe_actions":
            unsafe_actions,

        "unsafe_episode":
            int(unsafe_actions > 0),

        "poison_reached_ehr":
            int(
                ehr_entry is not None
                and ehr_entry.poisoned
            ),

        "insulin_poisoned":
            int(
                insulin_entry is not None
                and insulin_entry.poisoned
            ),

        "diet_poisoned":
            int(
                diet_entry is not None
                and diet_entry.poisoned
            ),

        "alert_poisoned":
            int(
                alert_entry is not None
                and alert_entry.poisoned
            )
    }

In [ ]:
test_result = run_attack_trial_v2(
    episodes.iloc[0].to_dict(),
    architecture="secure",
    attack_type="trusted_cgm"
)

print(test_result)

In [ ]:
architectures = [
    "global",
    "provenance",
    "scoped",
    "secure"
]

attack_types = [
    "untrusted",
    "trusted_cgm"
]

ablation_results = []

for architecture in architectures:

    for attack_type in attack_types:

        print(
            f"Running: {architecture:12s} | "
            f"{attack_type}"
        )

        for _, row in episodes.iterrows():

            result = run_attack_trial_v2(
                row.to_dict(),
                architecture=architecture,
                attack_type=attack_type
            )

            ablation_results.append(result)


ablation_df = pd.DataFrame(
    ablation_results
)

print("\nExperiment complete.")
print("Total trials:", len(ablation_df))

assert len(ablation_df) == 40000

print("✓ Expected 40,000 trials generated.")

In [ ]:
architectures = [
    "global",
    "provenance",
    "scoped",
    "secure"
]

attack_types = [
    "untrusted",
    "trusted_cgm"
]

ablation_results = []

for architecture in architectures:

    for attack_type in attack_types:

        print(
            f"{architecture:12s} | "
            f"{attack_type}"
        )

        for _, row in episodes.iterrows():

            result = run_attack_trial_v2(
                row.to_dict(),
                architecture=architecture,
                attack_type=attack_type
            )

            ablation_results.append(
                result
            )


ablation_df = pd.DataFrame(
    ablation_results
)

print(
    "\nTotal trials:",
    len(ablation_df)
)

In [ ]:
ablation_summary = []

for (
    architecture,
    attack_type
), group in ablation_df.groupby(
    [
        "architecture",
        "attack_type"
    ]
):

    total_possible = (
        group[
            "total_downstream_agents"
        ].sum()
    )

    affected = (
        group[
            "affected_agents"
        ].sum()
    )

    mpsr = (
        affected /
        total_possible
        if total_possible
        else 0
    )

    propagation = (
        group["affected_agents"]
        /
        group[
            "total_downstream_agents"
        ].replace(0, np.nan)
    ).mean()

    unsafe_rate = (
        group[
            "unsafe_episode"
        ].mean()
    )

    ehr_reach = (
        group[
            "poison_reached_ehr"
        ].mean()
    )

    ablation_summary.append({

        "Architecture":
            architecture,

        "Attack":
            attack_type,

        "MPSR":
            mpsr,

        "Propagation Reach":
            propagation,

        "EHR Reach":
            ehr_reach,

        "Unsafe Episode Rate":
            unsafe_rate,

        "Mean Unsafe Actions":
            group[
                "unsafe_actions"
            ].mean()
    })


ablation_summary_df = pd.DataFrame(
    ablation_summary
)

print(
    ablation_summary_df
    .sort_values(
        [
            "Attack",
            "Architecture"
        ]
    )
    .to_string(
        index=False
    )
)

ablation_summary_df.to_csv(
    "T6_2_ablation_results.csv",
    index=False
)

In [ ]:
class SecureTTLMemory(SecureClinicalMemory):

    def __init__(
        self,
        architecture="secure_ttl",
        ttl=3
    ):

        super().__init__(
            architecture=architecture,
            ttl=ttl
        )

        self.ttl = ttl


    def read(
        self,
        episode_id,
        key,
        current_time,
        requesting_agent=None
    ):

        candidates = [
            e for e in self.entries

            if e.episode_id == episode_id
            and e.key == key

            # Provenance
            and e.trusted

            # Integrity
            and create_hash(e)
            == e.integrity_hash

            # TTL
            and (
                current_time
                - e.timestamp
            ) <= self.ttl
        ]

        if not candidates:
            return None

        candidates = sorted(
            candidates,
            key=lambda x: x.timestamp
        )

        latest = candidates[-1]

        # Semantic glucose validation
        if (
            key == "glucose"
            and len(candidates) >= 2
        ):

            previous = candidates[-2]

            if not glucose_transition_valid(
                previous.value,
                latest.value
            ):

                return previous

        return latest

In [ ]:
def run_temporal_poison_trial(
    episode,
    architecture,
    ttl=3,
    max_time=10
):

    # -------------------------------
    # Memory architecture
    # -------------------------------

    if architecture == "global":

        memory = SharedMemory(
            architecture="global"
        )

    elif architecture == "provenance":

        memory = SharedMemory(
            architecture="provenance"
        )

    elif architecture == "secure":

        memory = SecureClinicalMemory(
            architecture="secure"
        )

    elif architecture == "secure_ttl":

        memory = SecureTTLMemory(
            ttl=ttl
        )

    else:

        raise ValueError(
            architecture
        )

    cgm = CGMAgent()
    insulin = InsulinAgent()
    diet = DietAgent()
    alert = AlertAgent()
    ehr = EHRAgent()

    # Legitimate baseline
    cgm.process(
        episode,
        memory,
        t=0
    )

    # Trusted CGM compromise
    inject_trusted_cgm_poison(
        memory,
        episode,
        timestamp=1
    )

    rows = []

    # Observe system over time
    for t in range(
        2,
        max_time + 1
    ):

        insulin_entry = insulin.process(
            episode,
            memory,
            t=t
        )

        diet_entry = diet.process(
            episode,
            memory,
            t=t
        )

        alert_entry = alert.process(
            episode,
            memory,
            t=t
        )

        ehr_entry = ehr.process(
            episode,
            memory,
            t=t
        )

        outputs = [
            insulin_entry,
            diet_entry,
            alert_entry,
            ehr_entry
        ]

        valid = [
            x for x in outputs
            if x is not None
        ]

        affected = sum(
            int(x.poisoned)
            for x in valid
        )

        rows.append({

            "episode_id":
                episode["episode_id"],

            "architecture":
                architecture,

            "ttl":
                ttl,

            "time":
                t,

            "affected_agents":
                affected,

            "available_agents":
                len(valid),

            "poison_active":
                int(affected > 0),

            "ehr_poisoned":
                int(
                    ehr_entry is not None
                    and ehr_entry.poisoned
                )
        })

    return rows

In [ ]:
temporal_episodes = episodes.sample(
    n=1000,
    random_state=42
)

TEMPORAL_ARCHITECTURES = [
    "global",
    "provenance",
    "secure",
    "secure_ttl"
]

temporal_results = []

for architecture in TEMPORAL_ARCHITECTURES:

    print(
        "Running:",
        architecture
    )

    for _, row in temporal_episodes.iterrows():

        results = run_temporal_poison_trial(
            row.to_dict(),
            architecture=architecture,
            ttl=3,
            max_time=10
        )

        temporal_results.extend(
            results
        )


temporal_df = pd.DataFrame(
    temporal_results
)

print(
    "\nRows:",
    len(temporal_df)
)

In [ ]:
persistence_table = (
    temporal_df
    .groupby(
        [
            "architecture",
            "time"
        ]
    )
    .agg(

        Poison_Active_Rate=(
            "poison_active",
            "mean"
        ),

        Mean_Affected_Agents=(
            "affected_agents",
            "mean"
        ),

        EHR_Poison_Rate=(
            "ehr_poisoned",
            "mean"
        )

    )
    .reset_index()
)

print(
    persistence_table.to_string(
        index=False
    )
)

In [ ]:
duration_rows = []

for (
    architecture,
    episode_id
), group in temporal_df.groupby(
    [
        "architecture",
        "episode_id"
    ]
):

    active_times = group.loc[
        group["poison_active"] == 1,
        "time"
    ]

    if len(active_times) == 0:

        duration = 0

    else:

        duration = (
            active_times.max()
            - 1
        )

    duration_rows.append({

        "architecture":
            architecture,

        "episode_id":
            episode_id,

        "persistence_duration":
            duration
    })


duration_df = pd.DataFrame(
    duration_rows
)

duration_summary = (
    duration_df
    .groupby("architecture")
    ["persistence_duration"]
    .agg(
        [
            "mean",
            "std",
            "median",
            "max"
        ]
    )
    .reset_index()
)

print(
    duration_summary.to_string(
        index=False
    )
)

In [ ]:
TTL_VALUES = [
    1,
    2,
    3,
    5,
    7
]

ttl_results = []

for ttl_value in TTL_VALUES:

    print(
        "Testing TTL:",
        ttl_value
    )

    for _, row in temporal_episodes.iterrows():

        results = run_temporal_poison_trial(
            row.to_dict(),
            architecture="secure_ttl",
            ttl=ttl_value,
            max_time=10
        )

        ttl_results.extend(
            results
        )


ttl_df = pd.DataFrame(
    ttl_results
)

In [ ]:
ttl_summary = (
    ttl_df
    .groupby("ttl")
    .agg(

        Poison_Active_Rate=(
            "poison_active",
            "mean"
        ),

        Mean_Affected_Agents=(
            "affected_agents",
            "mean"
        ),

        EHR_Poison_Rate=(
            "ehr_poisoned",
            "mean"
        )

    )
    .reset_index()
)

print(
    ttl_summary.to_string(
        index=False
    )
)

In [ ]:
def inject_bias_poison(
    memory,
    episode,
    timestamp,
    delta,
    trusted=True
):

    true_glucose = float(
        episode["true_glucose"]
    )

    poisoned_glucose = np.clip(
        true_glucose + delta,
        40,
        350
    )

    source = (
        "CGM_Agent"
        if trusted
        else "COMPROMISED_MEMORY_WRITER"
    )

    poisoned_entry = memory.write(
        episode_id=episode["episode_id"],
        key="glucose",
        value=float(poisoned_glucose),
        source_agent=source,
        timestamp=timestamp,
        trusted=trusted,
        poisoned=True
    )

    if poisoned_glucose < 70:
        fake_state = "LOW"

    elif poisoned_glucose > 180:
        fake_state = "HIGH"

    else:
        fake_state = "NORMAL"

    state_entry = memory.write(
        episode_id=episode["episode_id"],
        key="glucose_state",
        value=fake_state,
        source_agent=source,
        timestamp=timestamp,
        parent_id=poisoned_entry.entry_id,
        trusted=trusted,
        poisoned=True
    )

    return poisoned_entry, state_entry

In [ ]:
class SecureClinicalMemoryV2(
    SharedMemory
):

    def __init__(
        self,
        semantic_threshold=60
    ):

        super().__init__(
            architecture="secure_v2"
        )

        self.semantic_threshold = (
            semantic_threshold
        )

        self.rejections = 0


    def read(
        self,
        episode_id,
        key,
        current_time,
        requesting_agent=None
    ):

        candidates = [
            e for e in self.entries
            if e.episode_id == episode_id
            and e.key == key
            and e.trusted
            and create_hash(e)
            == e.integrity_hash
        ]

        if not candidates:
            return None

        candidates = sorted(
            candidates,
            key=lambda x: x.timestamp
        )

        latest = candidates[-1]

        if (
            key == "glucose"
            and len(candidates) >= 2
        ):

            previous = candidates[-2]

            difference = abs(
                float(latest.value)
                - float(previous.value)
            )

            if (
                difference
                > self.semantic_threshold
            ):

                self.rejections += 1

                return previous

        return latest

In [ ]:
def run_bias_trial(
    episode,
    delta,
    semantic_threshold
):

    memory = SecureClinicalMemoryV2(
        semantic_threshold=
            semantic_threshold
    )

    cgm = CGMAgent()
    insulin = InsulinAgent()
    diet = DietAgent()
    alert = AlertAgent()
    ehr = EHRAgent()

    # Legitimate measurement
    cgm.process(
        episode,
        memory,
        t=0
    )

    # Trusted compromised CGM
    inject_bias_poison(
        memory,
        episode,
        timestamp=1,
        delta=delta,
        trusted=True
    )

    insulin_entry = insulin.process(
        episode,
        memory,
        t=2
    )

    diet_entry = diet.process(
        episode,
        memory,
        t=3
    )

    alert_entry = alert.process(
        episode,
        memory,
        t=4
    )

    ehr_entry = ehr.process(
        episode,
        memory,
        t=5
    )

    expected = expected_actions(
        episode
    )

    unsafe = 0

    if (
        insulin_entry is not None
        and insulin_entry.value
        != expected["insulin_action"]
    ):
        unsafe += 1

    if (
        diet_entry is not None
        and diet_entry.value
        != expected["diet_action"]
    ):
        unsafe += 1

    if (
        alert_entry is not None
        and alert_entry.value
        != expected["clinical_alert"]
    ):
        unsafe += 1

    outputs = [
        insulin_entry,
        diet_entry,
        alert_entry,
        ehr_entry
    ]

    valid_outputs = [
        x for x in outputs
        if x is not None
    ]

    affected = sum(
        int(x.poisoned)
        for x in valid_outputs
    )

    return {

        "episode_id":
            episode["episode_id"],

        "delta":
            delta,

        "threshold":
            semantic_threshold,

        "true_glucose":
            episode["true_glucose"],

        "true_state":
            episode["true_state"],

        "affected_agents":
            affected,

        "unsafe_actions":
            unsafe,

        "unsafe_episode":
            int(unsafe > 0),

        "semantic_rejections":
            memory.rejections,

        "ehr_poisoned":
            int(
                ehr_entry is not None
                and ehr_entry.poisoned
            )
    }

In [ ]:
DELTAS = [
    -100,
    -60,
    -30,
    30,
    60,
    100
]

THRESHOLDS = [
    30,
    45,
    60,
    90
]

bias_results = []

for threshold in THRESHOLDS:

    for delta in DELTAS:

        print(
            f"Threshold={threshold:2d} | "
            f"Delta={delta:+4d}"
        )

        for _, row in episodes.iterrows():

            result = run_bias_trial(
                row.to_dict(),
                delta=delta,
                semantic_threshold=threshold
            )

            bias_results.append(
                result
            )


bias_df = pd.DataFrame(
    bias_results
)

print(
    "\nTotal trials:",
    len(bias_df)
)

In [ ]:
bias_summary = (
    bias_df
    .groupby(
        [
            "threshold",
            "delta"
        ]
    )
    .agg(

        Unsafe_Episode_Rate=(
            "unsafe_episode",
            "mean"
        ),

        Mean_Unsafe_Actions=(
            "unsafe_actions",
            "mean"
        ),

        Mean_Affected_Agents=(
            "affected_agents",
            "mean"
        ),

        EHR_Poison_Rate=(
            "ehr_poisoned",
            "mean"
        ),

        Semantic_Rejection_Rate=(
            "semantic_rejections",
            lambda x: (
                np.asarray(x) > 0
            ).mean()
        )
    )
    .reset_index()
)

print(
    bias_summary.to_string(
        index=False
    )
)

In [ ]:
def legitimate_followup(
    glucose,
    rng
):

    change = rng.normal(
        0,
        25
    )

    return float(
        np.clip(
            glucose + change,
            40,
            350
        )
    )

In [ ]:
utility_rows = []

rng = np.random.default_rng(123)

for threshold in THRESHOLDS:

    for _, episode in episodes.iterrows():

        original = float(
            episode["true_glucose"]
        )

        followup = legitimate_followup(
            original,
            rng
        )

        rejected = int(
            abs(
                followup - original
            ) > threshold
        )

        utility_rows.append({

            "threshold":
                threshold,

            "original":
                original,

            "followup":
                followup,

            "legitimate_rejected":
                rejected
        })


utility_df = pd.DataFrame(
    utility_rows
)

utility_summary = (
    utility_df
    .groupby("threshold")
    .agg(

        Legitimate_Rejection_Rate=(
            "legitimate_rejected",
            "mean"
        )

    )
    .reset_index()
)

print(
    utility_summary.to_string(
        index=False
    )
)

In [ ]:
def distance_to_boundary(g):

    boundaries = np.array(
        [70, 180, 250]
    )

    return np.min(
        np.abs(
            boundaries - g
        )
    )


bias_df[
    "boundary_distance"
] = bias_df[
    "true_glucose"
].apply(
    distance_to_boundary
)


bias_df[
    "boundary_group"
] = pd.cut(
    bias_df["boundary_distance"],
    bins=[
        -1,
        10,
        30,
        60,
        np.inf
    ],
    labels=[
        "≤10",
        "11–30",
        "31–60",
        ">60"
    ]
)


boundary_summary = (
    bias_df
    .groupby(
        "boundary_group",
        observed=True
    )
    .agg(

        Unsafe_Episode_Rate=(
            "unsafe_episode",
            "mean"
        ),

        Mean_Unsafe_Actions=(
            "unsafe_actions",
            "mean"
        )
    )
    .reset_index()
)

print(
    boundary_summary.to_string(
        index=False
    )
)